# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular *q-learning*.
You will first test your code with *random* and *greedy-epsilon* policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [1]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    """
    A 2D grid world simulator for a simple mobile rover.

    This class models a discrete-state Markov Decision Process (MDP) where a
    rover navigates a square grid to reach a terminal (goal) state.

    The state is an encoded integer representing (x, y, direction).
    The actions are turning left, maintaining direction, or turning right.
    Movement always includes one step forward in the current direction.
    """


    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        """
        Initializes the simulator grid, terminal state, and initial starting states.

        Args:
            resolution (int): The width and height of the square grid (e.g., 5 for a 5x5 grid).
        """    
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        """
        Encodes a (x, y, direction) tuple into a single integer state ID.

        The state ID is calculated as: (y * resolution + x) * 8 + direction.

        Args:
            x (int): The column coordinate (0 to resolution-1).
            y (int): The row coordinate (0 to resolution-1).
            direction (int): The current direction index (0 to 7).

        Returns:
            int: The unique integer state identifier.
        """


        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        """
        Decodes an integer state ID back into its (x, y, direction) components.

        Args:
            state (int): The unique integer state identifier.

        Returns:
            tuple: A tuple containing (x, y, direction).
        """

        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        """
        Returns the set of all possible actions for the rover.

        Actions are relative turns: -1 (left), 0 (straight), 1 (right).

        Args:
            state (int): The current state (unused, as actions are constant).

        Returns:
            list: A list of integers representing valid actions: [-1, 0, 1].
        """
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        """
        Calculates the next state and the reward based on the current state and action.

        The rover always moves one step forward in its current direction, and then
        its direction is updated based on the action. Movement is clamped to grid boundaries.

        Args:
            curr_state (int): The current state identifier.
            curr_action (int): The action taken (-1, 0, or 1).

        Returns:
            tuple: A tuple (next_reward, next_state), where:
                - next_reward (int): 1 if next_state is the terminal state, 0 otherwise.
                - next_state (int): The resulting state identifier.
        """
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x          = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y          = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction  = (curr_direction + curr_action) % 8

        next_state      = self.construct_state(next_x, next_y, next_direction)
        next_reward     = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        """
        Simulates a sequence of steps by executing a given policy function.

        The policy function is expected to take (state, available_actions) and
        return the chosen action. This is a generator function.

        Args:
            policy_func (function): A function (state, actions) -> action.
            max_steps (int): The maximum number of steps to simulate.

        Yields:
            tuple: (current_state, action, reward, next_state) for each step.
        """
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        """
        Randomly selects a starting state from the pre-computed list of legal initial states.

        Returns:
            int: A randomly selected initial state identifier.
        """
        return random.choice(self.initial_states)

In [2]:
simulator       = RoverSimulator(16)
initial_sample  = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1026


## Other Functions

In [3]:
def random_policy(state, actions):
    # Select a random action from the list of possible actions.
    return random.choice(actions)

In [4]:
def get_q_value(state, action, Q_table: dict):
    return Q_table.get((state, action), 0.0)

def max_q_value(state, actions, Q_table: dict):
    if state == simulator.terminal_state:
        return 0.0
    # Use the helper function to ensure unseen (state, action) pairs default to 0.0
    return max(get_q_value(state, a, Q_table) for a in actions)

In [5]:
def epsilon_greedy_policy(state, actions, epsilon = 0.25, Q_table: dict={}):
    if random.random() < epsilon:
        # Exploration: pick a random action
        return random_policy(state, actions)
    else:
        # Exploitation: pick the action with the highest Q-value
        q_values        = {action: get_q_value(state, action, Q_table) for action in actions}
        max_q           = max(q_values.values())
        best_actions    = [action for action, q_val in q_values.items() if q_val == max_q]
        return random.choice(best_actions)

In [6]:
# YOUR CHANGES HERE: Epsilon Scheduling

def linear_epsilon_schedule(episode_num, total_episodes, start_epsilon, end_epsilon):
    """
    Linearly decays epsilon from start_epsilon to end_epsilon over all episodes.
    """
    # Calculate the linear decay rate
    decay_rate = (start_epsilon - end_epsilon) / total_episodes
    
    # Calculate current epsilon
    current_epsilon = start_epsilon - decay_rate * episode_num
    
    # Clamp epsilon at the minimum (end_epsilon)
    return max(end_epsilon, current_epsilon)

# Example usage for testing and the next part:
# Epsilon goes from 1.0 down to 0.01 over 1000 episodes
# print(linear_epsilon_schedule(0, 1000, 1.0, 0.01))    # 1.0
# print(linear_epsilon_schedule(500, 1000, 1.0, 0.01))  # ~0.5
# print(linear_epsilon_schedule(1000, 1000, 1.0, 0.01)) # 0.01

## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [7]:
# YOUR CHANGES HERE

# def random_policy(state, actions):
#     # Select a random action from the list of possible actions.
#     return random.choice(actions)


Use the code below to test your random policy.
Then modify it to save the results in "`log-random.tsv`" with the columns `curr_state`, `curr_action`, `next_reward` and `next_state`.

In [8]:
# YOUR CHANGES HERE
import pandas as pd

log_data = []

# Run the rollout for a small number of steps (e.g., 32, as in the example)
# The max_steps should be chosen based on the full assignment requirements if they
# are not specified here. Using 32 as in the test example.
for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps = 32):
    # Print the output as requested in the original test code
    # print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

    log_data.append({
        "curr_state":    curr_state,
        "curr_action":   curr_action,
        "next_reward":   next_reward,
        "next_state":    next_state
    })

log_random_df = pd.DataFrame(log_data)
log_random_df

,curr_state,curr_action,next_reward,next_state
0,1093,-1,0,956
1,956,1,0,829
2,829,-1,0,692
3,692,-1,0,563
4,563,0,0,443
5,443,1,0,324
6,324,-1,0,195
7,195,-1,0,74
8,74,0,0,82
9,82,0,0,90


In [9]:
LOG_FILENAME = "log-random.tsv"

log_random_df.to_csv(LOG_FILENAME, sep='\t', index = False)

print(f"\nResults saved to {LOG_FILENAME}")



Results saved to log-random.tsv


Submit "`log-random.tsv`" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [10]:
import pandas as pd

Q_table         = {}
ALPHA           = 1.0  # Learning rate
GAMMA           = 0.9  # Discount factor
MAX_STEPS       = 1024
NUM_EPISODES    = 32



# --- Q-Learning Rollout and Data Collection ---
log_entries = [] 

print(f"Starting Q-Learning with Random Policy (Pandas method): {NUM_EPISODES} episodes of max {MAX_STEPS} steps.")

# YOUR CHANGES HERE

for episode in range(NUM_EPISODES):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=MAX_STEPS):
        
        s_a_key     = (curr_state, curr_action)
        old_q_value = get_q_value(curr_state, curr_action, Q_table)
        
        # Calculate Target: R + gamma * max_a' Q(s', a')
        next_state_actions  = simulator.get_actions(next_state)
        max_next_q          = max_q_value(next_state, next_state_actions, Q_table)
        target_value        = next_reward + GAMMA * max_next_q
        
        # Update rule: Q[s,a] = Target (since ALPHA=1)
        new_q_value         = target_value
        
        # Update Q-table
        Q_table[s_a_key]    = new_q_value
        
       
        log_entries.append({
            "curr_state":   curr_state, 
            "curr_action":  curr_action, 
            "next_reward":  next_reward, 
            "next_state":   next_state, 
            "old_value":    old_q_value, 
            "new_value":    new_q_value
        })


print(f"Q-Learning with Random Policy, total updates logged: {len(log_entries)}")


Starting Q-Learning with Random Policy (Pandas method): 32 episodes of max 1024 steps.
Q-Learning with Random Policy, total updates logged: 32768


In [11]:
# YOUR CHANGES HERE
df_log_2 = pd.DataFrame(log_entries)
df_log_2


,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,1927,-1,0,1926,0.0,0.0
1,1926,-1,0,1925,0.0,0.0
2,1925,0,0,1797,0.0,0.0
3,1797,-1,0,1668,0.0,0.0
4,1668,0,0,1540,0.0,0.0
...,...,...,...,...,...,...
32763,1088,1,0,1088,0.0,0.0
32764,1088,-1,0,1088,0.0,0.0
32765,1088,0,0,1088,0.0,0.0
32766,1088,-1,0,1088,0.0,0.0


Save each step in the simulator in a file "`q-random.tsv`" with columns `curr_state`, `curr_action`, `next_reward`, `next_state`, `old_value`, `new_value`.

In [12]:
# YOUR CHANGES HERE
LOG_FILENAME = "q-random.tsv" 


df_log_2.to_csv(LOG_FILENAME, sep = '\t', index = False)
print(f"Results saved to {LOG_FILENAME}.")

Results saved to q-random.tsv.


Submit "`q-random.tsv`" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "`q-greedy.tsv`" with columns `curr_state`, `curr_action`, `next_reward`, `next_state`, `old_value`, `new_value`.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [13]:


# --- Q-Learning with Epsilon-Greedy Rollout and Logging ---

# Reset Q-table as per the hint
Q_table.clear() 
EPSILON     = 0.25
log_entries = []

print(f"Starting Q-Learning with Epsilon-Greedy Policy (Epsilon={EPSILON}): {NUM_EPISODES} episodes of max {MAX_STEPS} steps.")

# YOUR CHANGES HERE

for episode in range(NUM_EPISODES):
    # Use  epsilon_greedy_policy for action selection
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps = MAX_STEPS):
        
        state_action_key = (curr_state, curr_action)
        old_q_value     = get_q_value(curr_state, curr_action, Q_table)

        # Find max Q for the next state
        next_state_actions = simulator.get_actions(next_state)
        max_next_q      = max_q_value(next_state, next_state_actions, Q_table)
        
        # Target = R + gamma * max_a' Q(s', a')
        target_value    = next_reward + GAMMA * max_next_q
        
        # Update rule: New Q = Target (since ALPHA=1)
        new_q_value     = target_value 
        
        # Update the Q-table
        Q_table[state_action_key] = new_q_value
        
        log_entries.append({
            "curr_state":   curr_state, 
            "curr_action":  curr_action, 
            "next_reward":  next_reward, 
            "next_state":   next_state, 
            "old_value":    old_q_value, 
            "new_value":    new_q_value
        })

df_log_3 = pd.DataFrame(log_entries)

print(f"\nQ-Learning with Epsilon-Greedy Policy, total updates logged: {len(log_entries)}")
print(f"Final Q-table size: {len(Q_table)}")

Starting Q-Learning with Epsilon-Greedy Policy (Epsilon=0.25): 32 episodes of max 1024 steps.

Q-Learning with Epsilon-Greedy Policy, total updates logged: 32768
Final Q-table size: 5295


In [14]:
df_log_3

,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,2,0,0,10,0.0,0.0
1,10,-1,0,17,0.0,0.0
2,17,-1,0,152,0.0,0.0
3,152,0,0,280,0.0,0.0
4,280,0,0,408,0.0,0.0
...,...,...,...,...,...,...
32763,1709,-1,0,1572,0.0,0.0
32764,1572,-1,0,1443,0.0,0.0
32765,1443,-1,0,1322,0.0,0.0
32766,1322,1,0,1331,0.0,0.0


In [15]:
LOG_FILENAME    = "q-greedy.tsv"
df_log_3.to_csv(LOG_FILENAME, sep='\t', index=False)
print(f"Saved to {LOG_FILENAME}.")

Saved to q-greedy.tsv.


Submit "`q-greedy.tsv`" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "`policy-greedy.tsv`" with columns `state` and `action`.

In [16]:
Q_table

{(2, 0): 0.0,
 (10, -1): 0.43046721000000016,
 (17, -1): 0.0,
 (152, 0): 0.0,
 (280, 0): 0.0,
 (408, 0): 0.0,
 (536, 1): 0.0,
 (665, 0): 0.0,
 (801, -1): 0.0,
 (936, 0): 0.0,
 (1064, 1): 0.0,
 (1193, -1): 0.0,
 (1328, -1): 0.0,
 (1463, -1): 0.0,
 (1582, 0): 0.0,
 (1574, 0): 0.0,
 (1566, -1): 0.0,
 (1557, 1): 0.0,
 (1422, 1): 0.0,
 (1415, 1): 0.2058911320946491,
 (1536, -1): 0.22876792454961012,
 (1671, 0): 0.2058911320946491,
 (1799, 0): 0.22876792454961012,
 (1927, 0): 0.22876792454961012,
 (1927, 1): 0.2058911320946491,
 (1920, 0): 0.2058911320946491,
 (1920, 1): 0.1853020188851842,
 (1921, 0): 0.1500946352969992,
 (1929, 0): 0.16677181699666577,
 (1937, 1): 0.0,
 (1946, 1): 0.0,
 (1955, 0): 0.0,
 (1835, -1): 0.0,
 (1714, 0): 0.0,
 (1722, 1): 0.0,
 (1731, -1): 0.0,
 (1610, -1): 0.0,
 (1617, 0): 0.0,
 (1753, 1): 0.0,
 (1890, 0): 0.0,
 (1898, 0): 0.0,
 (1906, -1): 0.0,
 (1913, -1): 0.0,
 (2040, 1): 0.0,
 (2041, -1): 0.0,
 (2040, 0): 0.0,
 (2040, -1): 0.0,
 (2047, 0): 0.0,
 (2039, 0): 0

In [17]:
# YOUR CHANGES HERE

# get all unique state IDs from the keys of the Q_table. (Identify all unique states with learned Q-values)
learned_states  = set([state_action[0] for state_action in Q_table.keys()])
print(f"Total unique states with learned Q-values: {len(learned_states)}")

# ---  get  Optimal Action for Each State ---
policy_entries  = []
actions         = [-1, 0, 1] # he possible actions for the Rover

for state in learned_states:
    
    # Check if  state is the terminal state (though it usually won't be in the policy)
    #  RoverSimulator logic handles the terminal state as a sink.
    
    # Find  Q-value for each possible action at this state
    #  use a default of 0.0 for actions not seen in this state, though in a well-explored Q-table, all actions should be present.
    q_values    = {action: Q_table.get((state, action), 0.0) for action in actions}
    
    # maximum Q-value
    max_q        = max(q_values.values())
    
    # get all actions that achieve the max Q-val (for tie-breaking)
    best_actions = [action for action, q_val in q_values.items() if q_val == max_q]
    
    # choose  best action randomly (to break ties)
    best_action  = random.choice(best_actions)
    
    policy_entries.append({
        "state":    state, 
        "action":   best_action
    })

print(f"Policy defined for {len(learned_states)} states.")


Total unique states with learned Q-values: 1968
Policy defined for 1968 states.


In [18]:
df_policy = pd.DataFrame(policy_entries)
df_policy

,state,action
0,2,-1
1,3,-1
2,4,-1
3,5,-1
4,6,-1
...,...,...
1963,2040,0
1964,2041,-1
1965,2042,0
1966,2043,-1


In [19]:

LOG_FILENAME = "policy-greedy.tsv"
df_policy.to_csv(LOG_FILENAME, sep='\t', index=False)

print(f"Results saved to {LOG_FILENAME}.")

Results saved to policy-greedy.tsv.


Submit "`policy-greedy.tsv`" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "`policy-optimal.tsv`" with columns `state` and `action`.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [20]:

Q_table.clear()   # reset Q-table
ALPHA           = 1.0  # Learning rate
GAMMA           = 0.9  # Discount factor
MAX_STEPS       = 1024
total_updates   = 0
NUM_EPISODES    = 5000 
EPSILON         = 0.25

print(f"Starting large Q-Learning run with epsilon-greedy (Epsilon={EPSILON}):\n{NUM_EPISODES} episodes of max {MAX_STEPS} steps.")
print(f"Alpha = {ALPHA}, Gamma = {GAMMA}")

# YOUR CHANGES HERE

for episode in range(NUM_EPISODES):
    # Use epsilon_greedy_policy for action selection
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps = MAX_STEPS):
        
        state_action_key    = (curr_state, curr_action)
        old_q_value         = get_q_value(curr_state, curr_action, Q_table)

        #  max Q for the next state
        next_state_actions  = simulator.get_actions(next_state)
        max_next_q          = max_q_value(next_state, next_state_actions, Q_table)

        # Target = R + gamma * max_a' Q(s', a')
        target_value    = next_reward + (GAMMA * max_next_q)
        
        # Update rule: New Q = Target (since ALPHA=1)
        new_q_value     = target_value 
        
        # Update the Q-table
        Q_table[state_action_key] = new_q_value
        total_updates             += 1
        
        # Hint: cut off rollouts for speed once they get the reward (next_reward == 1)
        if next_reward == 1:
            break
            

# ---  Extract Optimal Policy from Final Q-Table (Greedy Extraction) ---

learned_states  = set([state_action[0] for state_action in Q_table.keys()])
policy_entries  = []
actions         = [-1, 0, 1] 
best_actions    = []

print(f"\nTotal updates: {total_updates}. Q-table size: {len(Q_table)}")
print(f"Extracting policy for {len(learned_states)} unique states...")


for state in learned_states:
    
    # get Q-value for each possible action at this state
    q_values = {action: get_q_value(state, action, Q_table) for action in actions}
    
    # maximum Q-value
    max_q = max(q_values.values())
    
    # Find all actions that achieve this maximum value (for tie-breaking)
    # Use a small tolerance for floating point comparison for robustness
    for action, q_val in q_values.items():
        if abs(q_val - max_q) < 1e-6:
            best_actions.append(action)
    
    # choose from best actions randomly (to break ties)
    best_action = random.choice(best_actions)
    
    policy_entries.append({
        "state": state, 
        "action": best_action
    })
df_policy_optimal = pd.DataFrame(policy_entries)



Starting large Q-Learning run with epsilon-greedy (Epsilon=0.25):
5000 episodes of max 1024 steps.
Alpha = 1.0, Gamma = 0.9

Total updates: 4624327. Q-table size: 5985
Extracting policy for 1995 unique states...


In [21]:
df_policy_optimal

,state,action
0,0,1
1,1,1
2,2,0
3,3,-1
4,4,1
...,...,...
1990,2043,-1
1991,2044,1
1992,2045,0
1993,2046,-1


In [22]:
LOG_FILENAME = "policy-optimal.tsv"
print(f'Saving to {LOG_FILENAME}')
df_policy_optimal.to_csv(LOG_FILENAME, sep = '\t', index = False)

Saving to policy-optimal.tsv


Submit "`policy-optimal.tsv`" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

# Notes

<font color='plum'>

## Definition: Model-free Control


A model-free control method is any method for controlling a system that does not explicitly model system changes, such as transitions from state to state based on actions. 

Model-free control methods are still allowed to use modeling techniques; they just do not model the system behavior explicitly. For example, it is common to implement model-free control with a model for state-action values or approximate policies. Both approaches using state-action values and policies may implicitly model some of the state-to-state transitions. Neural networks, in particular, are known to develop internal representations that may relate to important state changes. However, that modeling is incidental and not explicitly designed or optimized, so these methods are still considered model-free. 

#### Motivations from Simplicity
Focusing on state-action values or approximate policies can result in a simpler system. Given a policy, the action distribution can be directly sampled to pick an action. Or, given the state-action values for all actions at the current state, the action with the highest expected value can be chosen.

How would state values and a model of state transitions be used? For each action at the current state, predict the distribution of resulting states and then take the average to calculate all the state-action values. Then the action with the best value is selected. But this is just a more complicated version of using state-action values. And it fails if either the state values or the model of state transitions is missing. 

#### Motivations from Complexity
Another argument for the use of model-free methods is that they work better with the same model resources. Suppose that the set of all states is ﻿calligraphic S﻿, and the set of all actions is ﻿calligraphic A﻿. Then just storing transition probabilities takes ﻿vertical line calligraphic S vertical line squared vertical line calligraphic A vertical line﻿ entries. Meanwhile, storing state-action values takes just ﻿vertical line calligraphic S vertical line vertical line calligraphic A vertical line﻿ entries. When ﻿calligraphic S﻿ is large, that is a big difference, and many of the value function computations will scale with the number of entries. These issues still remain if the value function is approximated, e.g., by a neural network, since a similar amount of space is required in general. 

#### Motivations from Stability
Our last motivation for model-free control is stability. A model-based policy will be optimal for that model, but if the model is inaccurate, the policy may be very different from the true optimal policy (Atkeson & Santamaria, 1997). While we saw some methods for robust planning with inaccurate models last week, they at least knew the form of the true model and updated the model as the agent ran to correct those inaccuracies. Model-free control is often applied when the form of the true system dynamics is unknown, so those model inaccuracies will be hard to reason about. 

## Proportional-Integral-Derivative (PID) Control

Proportional-integral-derivative (PID) controllers are our first example of **model-free control** and one of the oldest automated control methods. They were originally designed for steering and speed control (Minorsky, 1922) and are now ubiquitous in thermostats in our homes and cruise control in our cars.

> Anecdotally, more than 95% of all controllers are PID controllers.

PID controllers are common because they are simple and do not require a model of the system that they control. Their behavior can be summed up simply as:

> **Trying harder the farther they are from their target.**

## How Does a Thermostat Work?

In simplest terms:

- If the area it controls is **too cold**, it turns on the **heat**.
- If the area it controls is **too hot**, it turns on the **air conditioning**.

Some thermostats:
- Control only heat
- Control only air conditioning
- Control both (but only one at a time)

### How much does the thermostat turn on the heat or air conditioning?

That depends on **how far off** the current temperature is from the target temperature.

- If it's **very far off**, it should work **harder**.
- If it's been far off for a **while**, it should **increase the effort**.
- If it's **moving farther** from the target, it should **increase the effort even more**.

But exactly **how much effort** should it use? That’s where **PID control** comes in.

--------------
## What Is PID Control?

### Definition: Process Variable

A **process variable** is a variable in a system that is monitored or controlled.

- Example: In a one-dimensional quadcopter, the process variable is the **height**.
- Denoted as: $$y(t)$$

### Definition: Setpoint

A **setpoint** is the desired value of a process variable.

- The controller's goal is to make the process variable match the setpoint.
- Denoted as: $$r(t)$$
- Often, $$r(t)$$ is a **constant**.

### Definition: Error Value

The **error value** is the difference between the process variable and its setpoint.

- Denoted as: $$e(t)$$
- Formula: $$e(t) = r(t) - y(t)$$


### Definition: PID Control

A **PID controller** takes in the error value and returns a **linear combination** of:

1. The **current error value**
2. The **integral** of the error over time
3. The **derivative** of the error

The control action $$u(t)$$ is given by:

$$
u(t) = K_P e(t) + K_I \int_{-\infty}^{t} e(t) \, dt + K_D \frac{d}{dt} e(t)
$$

Where:
- $$K_P$$ = Proportional gain
- $$K_I$$ = Integral gain
- $$K_D$$ = Derivative gain


## What Kinds of Problems Use PID Control?

PID controllers are used for problems with **one process variable** to manage.

### Common Applications:
- Thermostats
- Cruise control (speed)
- Depth control (e.g., submarines)
- Other simple systems needing stability

They are popular because:
- They are **flexible**
- They work **without a model** of the system

### Why are they flexible?

- **Integral term**: Corrects **persistent errors**
- **Derivative term**: Reacts to **rapid changes**

## How Are PID Controllers Configured?

PID constants are often configured **empirically**:

- Initialized from similar systems
- Tuned to balance:
  - **Fast error correction**
  - **System stability**

### Trade-off:

- Large $$K_P$$ → Fast correction but **oscillations**
- Small $$K_P$$ → Slower correction but **more stable**


---------

## Q-Learning and Tabular Learning

If the dynamics of a system are unknown but the number of state-action pairs is small enough, a model-free technique called **Q-learning** can be used (Watkins, 1989).

Recall the Bellman equations from Week 5:

$$
v^*(S_t) = \max_{a \in \mathcal{A}(S_t)} \left( \mathbb{E}[R_{t+1} \mid S_t, A_t = a] + \gamma \mathbb{E}[v^*(S_{t+1}) \mid S_t, A_t = a] \right)
$$

Or equivalently:

$$
q^*(S_t, a) = \mathbb{E}[R_{t+1} \mid S_t, A_t = a] + \gamma \mathbb{E}[v^*(S_{t+1}) \mid S_t, A_t = a]
$$

$$
v^*(S_t) = \max_{a \in \mathcal{A}(S_t)} q^*(S_t, a)
$$

- The function **\( v^* \)** returns the optimal value of a state.
- The function **\( q^* \)** returns the optimal value of a state-action pair.


### Definition: Q-Learning

**Q-learning** is any process that learns the optimal state-action value function \( q^* \).



#### Definition: Tabular Learning

**Tabular learning** is a process where values to be learned are mapped to table entries, and the table entries are updated during the learning process.

- The value iteration algorithms covered in Week 6 for finite Markov decision processes count as tabular learning if the results of each iteration are stored in the same table.
- The original Q-learning algorithm used tabular learning.
- It was noted that Q-learning may not converge to \( q^* \) if a different representation is used (Watkins, 1989).


### The Q-Learning Algorithm

The Q-Learning algorithm works as follows (Watkins et al., 1992):

1. **Initialize** a table \( Q[s, a] \) with an entry for every state \( s \) and action \( a \).
2. For all final states \( s_f \) without any allowed actions, initialize \( Q[s_f, a] \) to the reward associated with \( s_f \), if any, for all actions \( a \).
3. Initialize the remaining entries to 0.
4. Repeatedly sample a non-final state \( s_n \), pick action \( a_n \), observe the resulting payoff \( r_n \) and next state \( t_n \), and update \( Q[s_n, a] \) as follows:

$$
Q[s_n, a] = (1 - \alpha_n) Q[S_n, a] + \alpha_n \left( r_n + \gamma \max_b Q[t_n, b] \right)
$$

- alpha_n is the **learning rate**.
- gamma is the **discount factor** for diminishing returns.

If \( \alpha_n \) is a constant, this update computes an **exponential moving average** of the sampled transitions from \( s_n \) and \( a \).

### Should \alpha_n\ Be a Constant?

- If the system is **deterministic**, set \( \alpha_n = 1 \).
- If the system is **probabilistic**, \( \alpha_n \) should **slowly decline** with the number of times that \( s_n \) and \( a \) have been sampled.

See *Watkins & Dayan (1992)* for details on how \( \alpha_n \) should behave to guarantee convergence of \( Q[s_n, a] \) to \( q^*(s_n, a) \).

> In practice, many practitioners ignore this technical condition and set \( \alpha_n \) to a constant such as 0.1.

### How Should the Sampling Process Work?

- Technically, **some random sampling** of \( s_n \) and \( a \) is needed to guarantee convergence to \( q^* \) over an infinite time scale.
- In practice, **random rollouts** are used to sample from the beginning of the process.
- Each action during the rollout is used as a sample.
- These rollouts can pick the **best action** according to \( Q[s_n, a] \) most of the time but must include **some random sampling** (e.g., **ϵ-greedy**) to guarantee convergence.



---

## Limits of Tabular Q-Learning

The main limit of **tabular Q-learning** is that the table must be large enough to store all possible **state-action pairs**. This restricts tabular Q-learning to problems with **finite states** and **finite actions**.

Additionally, there is **no general bound** on the number of iterations required for Q-learning to reach optimality. In practice, the algorithm is often stopped when the state-action values appear to stabilize, even if they are still **suboptimal**.

---

## Bias of Q-Learning Before Convergence

Q-learning **will converge** to the optimal \( q^* \) values over time. However, in the **short term**, it tends to **overestimate** values due to the use of the **max** operator in its update rule.

This overestimation is common in **probabilistic environments** and is not unique to Q-learning. Here's why it happens:

- A **lucky high reward** or a transition to a **high-value state** can cause an **overestimate** of the action value.
- That overestimated action value then inflates the **state value**.
- These inflated values can **cascade** to other actions and states, compounding the bias.

### Analogy to Bandit Algorithms

This behavior is similar to **bandit algorithms** (Week 3), where the agent chooses the action (arm) that currently looks best. However, unlike bandit algorithms:

- Q-learning’s overestimates can **propagate** to other state-action pairs.
- These overestimates **linger longer**, slowing down convergence.

### Real-World Example

Imagine training a **driving agent**:

- It steps on the gas to beat a yellow light and receives a high reward.
- A human might judge this as a poor decision, but Q-learning records a **high value** for that action.
- Later, the agent might step on the gas earlier at similar intersections.
- Even if a simulated accident later reduces the value of the original action, the **derived values** from that action may **persist**, leading to poor decisions.

> Hopefully, all this learning occurs in **simulated environments**, not on real streets.

---------
## Avoiding Overestimation: Double Q-Learning

**Double Q-learning** addresses this issue by using **two tables** of estimates in parallel (Hasselt, 2010):

- For each training step:
  - One table is used to **select** the action with the maximum value.
  - The **other table** is used to **evaluate** the expected value of that action.
  - The first table is then **updated** using the value from the second table.

This approach:

- Introduces a **bias** in the first table due to the max operator.
- But **corrects** that bias using the second table’s estimate.
- **Alternates** the roles of the tables during training.

While **Double Q-learning** does **not eliminate** overestimation entirely, it **significantly reduces** it.
